# Capstone — From processed data to place cells

This is where the week comes together. Four instruments recorded the same
animal, and we now fuse them into a single, analyzable picture of *where the
animal was* and *what each neuron did there*:

| Stream | Tool | What it gives us |
|---|---|---|
| calcium traces + footprints | **Minian** | `C.zarr`, `A.zarr`, `max_proj.zarr` |
| deconvolved neural activity | **CaTune / CaDecon** (calab) | `deconv_out/activity.npy` |
| animal position | **eztrack** | `position.csv` |
| the clocks | **miniscope + behavior DAQ** | neural & behavior timestamps |

**CaMAP** does the fusion and the place-cell analysis. We run the pipeline
*live*, stopping at every step to **look at what it produced** — because most
real-world mistakes in this kind of analysis are invisible in the numbers and
obvious in a plot.

Three ideas run through the whole notebook, and each gets called out where it
bites:

1. **Combining two instruments** — two cameras, two coordinate systems, two
   sampling rates. They have to be reconciled before any neuron-vs-place claim
   means anything.
2. **Timestamping & sync** — the neural clock and the behavior clock are
   *different clocks*. Getting position onto the neural clock correctly is the
   single most important step, and the easiest to get silently wrong.
3. **Place-cell analysis** — what actually makes a cell a "place cell," and how
   each metric (rate map → spatial information → stability → place field)
   defends against a different way of being fooled.

> Interactive widgets (the unit browser at the end) need Jupyter Lab:
> `jupyter lab` from the repo root, then pick the **Workshop** kernel.

## Setup & data paths

Pick the session. Inputs come from your own upstream runs **or** from Zenodo —
`python scripts/get_data.py` populates `data/sessions/<name>/` local-first (see
`data/README.md`). The capstone reads the same paths either way.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # make workshop_glue importable

SESSION = "prerecorded"        # or "live" for the workshop recording
SESS = Path(f"../data/sessions/{SESSION}")

MINIAN_OUT  = SESS / "minian_out"                       # Minian zarr output
DECONV_OUT  = SESS / "deconv_out"                        # calab: CaTune / CaDecon
EZTRACK_CSV = SESS / "eztrack_out" / "position.csv"      # raw eztrack output
NEURAL_TS   = SESS / "raw" / "neural_timestamp.csv"      # miniscope DAQ clock
BEHAVIOR_TS = SESS / "raw" / "behavior_timestamp.csv"    # behavior DAQ clock

CONFIG     = Path("config/arena_config.yaml")   # analysis parameters
DATA_PATHS = Path("config/data_paths.yaml")     # per-session file paths

# Sanity check: if a stage is missing, fetch it.
missing = [p for p in (MINIAN_OUT, DECONV_OUT, EZTRACK_CSV, NEURAL_TS, BEHAVIOR_TS) if not p.exists()]
for p in missing:
    print(f"MISSING {p}")
if missing:
    print(f"\n-> run: python scripts/get_data.py --session {SESSION}")
else:
    print("All inputs present.")

## 1 · Glue: eztrack → DeepLabCut format

> **Combining-instruments note #1 — file conventions.**
> eztrack writes a flat `frame, x, y` table. CaMAP's behavior loader expects a
> DeepLabCut-style CSV with a 3-row `scorer / bodyparts / coords` header, and
> picks a column by `(scorer, bodypart, coord)`. Neither tool is wrong — they
> just disagree on format. A thin converter is all it takes, and getting it
> right is the difference between "tracked the LED" and "tracked nothing."

`bodypart` here **must** match `behavior.bodypart` in `config/data_paths.yaml`.

In [ ]:
from workshop_glue import eztrack_to_camap, miniscope_timestamps_to_camap

# ezTrack flat CSV -> the position CSV CaMAP reads (3-row header)
pos_csv = eztrack_to_camap(
    EZTRACK_CSV,
    SESS / "eztrack_out" / "position_camap.csv",   # what data_paths.yaml points at
    bodypart="LED",                                 # matches behavior.bodypart
)
print("Wrote", pos_csv)

# Miniscope DAQ timestamps -> CaMAP schema (seconds), for neural + behavior
for dev in ("neural", "behavior"):
    ts = miniscope_timestamps_to_camap(
        SESS / "raw" / f"{dev}_timestamp.csv",
        SESS / "raw" / f"{dev}_timestamp_camap.csv",
    )
    print("Wrote", ts)

## 2 · `load()` — what each instrument gave us

`from_yaml` reads the analysis config + the data-paths config and returns an
`ArenaDataset` (chosen automatically from `behavior.type: arena`). `load()`
then pulls in the neural traces, the spatial footprints, the max-projection
image, and the raw behavior trajectory.

> **Combining-instruments note #2 — two coordinate systems.**
> The footprints and traces live in *miniscope pixel* space; the trajectory
> lives in *behavior-camera pixel* space. They are never the same pixels. What
> ties them together is **time**, not space — which is why the sync step
> (coming up) carries all the weight.

We look at the neural side two ways: the footprints over the max projection
(*are these real cells?*) and a handful of raw calcium traces (*do they look
like calcium?*).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from camap.dataset import BaseCaMAPDataset

ds = BaseCaMAPDataset.from_yaml(CONFIG, DATA_PATHS)
ds.load()

n_units  = ds.traces.sizes["unit_id"]
n_frames = ds.traces.sizes["frame"]
print(f"{n_units} units x {n_frames} neural frames")
print(f"raw trajectory: {len(ds.trajectory)} behavior samples")
print(f"config: bins={ds.spatial.bins}, spatial_sigma={ds.spatial.spatial_sigma}, "
      f"n_shuffles={ds.spatial.n_shuffles}, p_threshold={ds.spatial.p_value_threshold}")

In [ ]:
# Footprints over the max projection — a quick "are these cells?" sanity check.
from camap.visualization import plot_footprints

if ds.footprints is not None and ds.max_proj is not None:
    plot_footprints(ds.max_proj, ds.footprints)
    plt.show()
else:
    print("No footprints / max projection in this session (neural-only check skipped).")

In [ ]:
# A few raw Minian calcium traces — slow rise, exponential decay = calcium.
fig, ax = plt.subplots(figsize=(11, 3.5))
t = np.arange(n_frames) / ds.neural_fps
for i, uid in enumerate(ds.traces["unit_id"].values[:5]):
    ax.plot(t, ds.traces.sel(unit_id=uid).values + i * 1.0, lw=0.6)
ax.set_xlabel("time (s)"); ax.set_ylabel("unit (offset)"); ax.set_yticks([])
ax.set_title("Example Minian denoised calcium traces (C)")
plt.show()

## 3 · `preprocess_behavior()` — clean the trajectory, get into mm

Raw tracking is noisy and lives in pixels. `preprocess_behavior()` runs, in
order: **perspective correction** → **Hampel jump removal** (kills tracking
teleports) → **arena clipping** → **pixel → millimetre conversion**.

> **Combining-instruments note #3 — calibration is physical.**
> The px→mm conversion uses `arena_bounds`, `arena_size_mm`, and the camera /
> tracking-point heights from `config/data_paths.yaml`. If those are wrong,
> every speed and every distance downstream is wrong — by a constant factor, so
> nothing *looks* broken. Calibrate deliberately. (Speeds report as `mm/s` only
> when calibration succeeded; otherwise CaMAP falls back to `px/s`.)

`plot_preprocess_steps` shows the trajectory at each stage so you can see what
each cleaning step actually removed.

In [ ]:
ds.preprocess_behavior()

from camap.visualization import plot_preprocess_steps

dbcfg = ds.data_cfg.behavior
if getattr(dbcfg, "arena_size_mm", None) is not None:
    plot_preprocess_steps(ds._preprocess_steps, dbcfg.arena_size_mm)
    plt.show()
else:
    print("arena_size_mm not set in data_paths.yaml -> skipping the mm-space step plot.")

print("calibration:", f"{ds.mm_per_px:.4f} mm/px" if ds.mm_per_px else "uncalibrated (px units)")

## 4 · Inject deconvolved neural activity (calab: CaTune / CaDecon)

Deconvolution is an **explicit upstream stage** in this workshop — Minian's own
deconvolution is skipped, and calab (CaTune or CaDecon) turned the denoised
traces into a per-unit estimate of neural activity, saved as
`deconv_out/activity.npy`. So instead of CaMAP's built-in OASIS
(`ds.deconvolve()`), the glue loads that file and sets `ds.good_unit_ids` /
`ds.S_list` directly.

The activity rows are aligned 1:1, **in order**, with the Minian `C` traces fed
to calab — i.e. the same order as `ds.traces`'s `unit_id` axis. The injector
asserts both the unit count and the frame count match, so a mismatch fails loud
here rather than producing a quietly-wrong analysis later.

In [ ]:
from workshop_glue import inject_deconv_activity

inject_deconv_activity(ds, DECONV_OUT)
print(f"Injected deconvolved activity for {len(ds.good_unit_ids)} units "
      f"({len(ds.S_list[0])} frames each)")

# Fallback — CaMAP's own OASIS, using the oasis block in arena_config.yaml:
# from tqdm.auto import tqdm
# ds.deconvolve(progress_bar=tqdm)

## 5 · Two clocks — timestamping & sync (`match_events()`)

**This is the conceptual heart of the whole pipeline.** The miniscope and the
behavior camera are independent devices on independent clocks. Frame *k* of the
neural recording and frame *k* of the behavior video are **not** the same
moment in time — the cameras run at different rates, start at slightly different
times, and drop frames independently.

`match_events()` resolves this by treating the **neural timestamps as the
canonical clock** and *interpolating* the behavior position onto them:

1. **Validate the neural clock.** Anomalous timestamps (duplicates, backward
   jumps, gaps `> 2×` the median interval) are detected and dropped, so they
   can't corrupt the interpolation.
2. **Check the clocks overlap.** If the neural rate is `> 5×` the behavior rate,
   or the time windows don't overlap, that's an error — not something to paper
   over.
3. **Linearly interpolate** behavior `(x, y)` onto each surviving neural
   timestamp. Neural frames outside the behavior window get `NaN`.
4. **Compute speed** at the neural rate with a centered window.

The result is `ds.canonical`: **one row per neural frame**, carrying the
animal's position at that instant plus each unit's activity (`s_unit_*`). Every
downstream metric reads from this one table.

> **Why this matters:** if you instead just paired frame *k* with frame *k*, a
> few dropped frames early in the session would shift position relative to
> activity for the *entire rest of the recording* — and the place fields would
> smear toward noise. The bug wouldn't throw; it would just quietly lower every
> spatial-information score.

In [ ]:
ds.match_events()

print("canonical table:", ds.canonical.shape, "(one row per neural frame)")
print("columns:", [c for c in ds.canonical.columns if not c.startswith("s_unit_")],
      f"+ {sum(c.startswith('s_unit_') for c in ds.canonical.columns)} s_unit_* columns")
ds.canonical.head()

### Inspect the clocks directly

`plot_timestamp_diagnostics` plots frame index vs. unix time for each stream. A
**healthy recording is a near-straight ramp**: no flat segments (duplicate
timestamps), no backward steps (non-monotonic, marked with red ×), and the two
panels should have *similar slopes* if the rates are similar. Kinks, plateaus,
or wildly different slopes here explain almost every weird downstream result —
look before you analyze.

In [ ]:
from camap.visualization import plot_timestamp_diagnostics

plot_timestamp_diagnostics(ds.trajectory_raw, ds.canonical)
plt.show()

## 6 · `compute_occupancy()` — where the animal spent time

A rate map is "activity per unit *time spent*," so we first need the time spent
in each spatial bin: the **occupancy map**. Bins the animal barely visited can't
support a reliable rate estimate, so `min_occupancy` marks them invalid (they'll
show as holes, not as zeros).

> **Combining-instruments note #4 — speed filtering.**
> Place coding is about *navigation*, so frames where the animal is essentially
> still (below `speed_threshold`) are dropped for the rate maps. This also
> sidesteps a classic artifact: a cell that fires while the animal rests in one
> corner would otherwise look like a place cell for that corner. `match_events`
> already built the speed-filtered view (`ds.trajectory_filtered`,
> `ds.event_place`) off the canonical table.

In [ ]:
ds.compute_occupancy()

from camap.visualization import plot_occupancy_preview

scfg = ds.spatial
plot_occupancy_preview(
    ds.trajectory_filtered,
    ds.occupancy_time, ds.valid_mask, ds.x_edges, ds.y_edges,
    behavior_fps=ds.neural_fps,
    stability_splits=scfg.stability_splits,
    block_shift=scfg.block_shift,
    min_occupancy=scfg.min_occupancy,
    spatial_sigma=scfg.spatial_sigma,
)
plt.show()

In [ ]:
# Speed distribution + the kept (moving) trajectory: shows what the speed
# threshold actually removed.
from camap.visualization import plot_behavior_preview

plot_behavior_preview(
    ds.canonical, ds.trajectory_filtered,
    ds.cfg.behavior.speed_threshold,
    speed_unit="mm/s" if ds.mm_per_px else "px/s",
)
plt.show()

## 7 · What makes a place cell? (the four questions)

Before running the analysis on every unit, it's worth being explicit about what
we're testing. A cell earns "place cell" by answering four questions, each
guarding against a different way of being fooled:

1. **Rate map** — *where does it fire?* Activity binned by position and divided
   by occupancy, then lightly smoothed. This is the picture; the rest are tests
   *on* this picture.
2. **Spatial information (Skaggs, bits/spike)** — *is the tuning more spatial
   than chance?* A cell that fires everywhere has low SI; a cell concentrated in
   one spot has high SI. But SI alone is biased upward by low firing rates and
   uneven sampling — so we need a null.
3. **Shuffle null → p-value** — *could this SI arise from random timing?* We
   circularly shift the activity relative to position many times (preserving the
   activity's own temporal structure, destroying its bond to place) and rebuild
   the SI each time. The observed SI must beat that null distribution. A
   `min_shift` prevents trivially-small shifts from leaking real structure into
   the null.
4. **Stability** — *is the field the same across the session?* The map from the
   first half (or interleaved blocks) must correlate with the second. A
   transient burst can fake high SI once; it can't fake a *stable* field. A cell
   is a place cell only if it is **both significant and stable**.

`analyze_units()` runs all four for every unit; then we open one unit up to see
the machinery.

## 8 · `analyze_units()` — all units + population summary

In [ ]:
from tqdm.auto import tqdm

# The per-unit spatial-information shuffle test (n_shuffles in arena_config.yaml)
# is the slow step: ~40-50 min for 708 units x 1000 shuffles, single-threaded.
# analyze_units(n_workers=N) parallelizes it across processes -- a solid win on
# Linux/macOS, but on Windows the spawn+pickle overhead caps it near ~1.3x (and
# ProcessPool can hang under Jupyter), so it's left serial here. For a quicker,
# less precise pass, lower n_shuffles in arena_config.yaml.
ds.analyze_units(progress_bar=tqdm)

s = ds.summary()
print(f"{s['n_total']} units | {s['n_sig']} significant ({s['pct_sig']}%) | "
      f"{s['n_stable']} stable ({s['pct_stable']}%) | "
      f"{s['n_place_cells']} place cells ({s['pct_place_cells']}%)")

In [ ]:
# Population view: significance vs stability, SI vs Fisher-z, and the density
# contour separating place cells from the rest.
from camap.visualization import plot_summary_scatter, plot_diagnostics

plot_summary_scatter(
    ds.unit_results,
    p_value_threshold=ds.p_value_threshold,
    n_shuffles=ds.spatial.n_shuffles,
    min_shift_seconds=ds.spatial.min_shift_seconds,
)
plt.show()

plot_diagnostics(ds.unit_results, p_value_threshold=ds.p_value_threshold)
plt.show()

## 9 · Metrics deep-dive — one unit, all four questions

Now we open the hood on a single unit and reproduce each metric from the stored
`UnitResult`, so the math is visible rather than hidden inside `analyze_units()`.
We pick the most spatially-informative unit available.

In [ ]:
# Pick the example unit: highest-SI place cell if any, else highest-SI unit.
place = ds.place_cells()
pool = place if place else ds.unit_results
UID = max(pool, key=lambda u: ds.unit_results[u].si)
res = ds.unit_results[UID]
print(f"Example unit {UID}: SI={res.si:.3f} bits/spike, p={res.p_val:.4f}, "
      f"{len(res.unit_data)} events, "
      f"{'PLACE CELL' if UID in place else 'not a place cell'}")

### Q1 — the rate map (built from scratch vs. stored)

To show there's no magic, we rebuild the smoothed rate map directly from this
unit's events + the occupancy map with `compute_rate_map`, and compare it to the
one `analyze_units` stored. They should be identical.

In [ ]:
from camap.analysis.spatial_2d import compute_rate_map

rm_manual = compute_rate_map(
    res.unit_data,                     # this unit's speed-filtered events (x, y, s)
    ds.occupancy_time, ds.valid_mask, ds.x_edges, ds.y_edges,
    spatial_sigma=ds.spatial.spatial_sigma,
    normalize=False,                   # firing-rate units, like the stored map
)
rm_stored = res.rate_map_smoothed

ext = [ds.x_edges[0], ds.x_edges[-1], ds.y_edges[0], ds.y_edges[-1]]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, m, ttl in [
    (axes[0], res.rate_map_raw, "raw rate map (events / occupancy)"),
    (axes[1], rm_manual,        "smoothed — compute_rate_map()"),
    (axes[2], rm_stored,        "smoothed — stored in UnitResult"),
]:
    disp = m.copy(); disp[~ds.valid_mask] = np.nan
    im = ax.imshow(disp.T, origin="lower", extent=ext, cmap="inferno", aspect="equal")
    ax.set_title(ttl, fontsize=9); ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, label="rate (a.u./s)")
plt.tight_layout(); plt.show()

print("manual vs stored max abs diff:",
      float(np.nanmax(np.abs(rm_manual - rm_stored))))

### Q2 + Q3 — spatial information and the shuffle null

`analyze_units` already computed the SI and the full shuffle null distribution
(`res.shuffled_sis`). We plot that null with the observed SI marked, and read
the p-value off as the fraction of shuffles that beat the real cell.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(res.shuffled_sis, bins=40, color="0.7", edgecolor="none", label="shuffled null")
ax.axvline(res.si, color="crimson", lw=2, label=f"observed SI = {res.si:.3f}")
p_emp = (np.sum(res.shuffled_sis >= res.si) + 1) / (len(res.shuffled_sis) + 1)
ax.set_xlabel("spatial information (bits/spike)"); ax.set_ylabel("# shuffles")
ax.set_title(f"Shuffle null  ·  stored p={res.p_val:.4f}  ·  empirical p={p_emp:.4f}  "
             f"(n_shuffles={len(res.shuffled_sis)})")
ax.legend()
plt.tight_layout(); plt.show()

### Q4 — stability across the session

Each configured split (e.g. `[2, 10]`) divides the session into contiguous
blocks, builds a rate map from the odd blocks and another from the even blocks,
and correlates them. `n=2` is the classic first-half/second-half test (sensitive
to slow drift); larger `n` interleaves blocks so drift averages out (a finer
timescale test). A place cell's two maps look alike.

In [ ]:
splits = res.stability_splits
n_sp = len(splits)
fig, axes = plt.subplots(n_sp, 2, figsize=(7, 3.4 * n_sp), squeeze=False)
for r, sp in enumerate(splits):
    for c, (m, half) in enumerate([(sp.rate_map_first, "odd blocks"),
                                   (sp.rate_map_second, "even blocks")]):
        disp = m.copy(); disp[~ds.valid_mask] = np.nan
        ax = axes[r, c]
        ax.imshow(disp.T, origin="lower", extent=ext, cmap="inferno", aspect="equal")
        ax.axis("off")
        ax.set_title(f"split n={sp.n_split_blocks}, {half}", fontsize=9)
    axes[r, 0].set_ylabel(f"r={sp.corr:.2f}", fontsize=10)
fig.suptitle(f"Unit {UID} stability — correlation between halves", y=1.0)
plt.tight_layout(); plt.show()

for sp in splits:
    flag = "stable" if (np.isfinite(sp.p_val) and sp.p_val < ds.p_value_threshold) else "—"
    print(f"  split n={sp.n_split_blocks:>2}: r={sp.corr:+.3f}, "
          f"p={sp.p_val:.4f}  [{flag}]")
print(f"is_stable (all splits pass): {res.is_stable(ds.p_value_threshold)}")

### Putting it together — the place field

A "place field" is the contiguous region where the rate clears a threshold
*and* exceeds the per-bin shuffle ceiling (`shuffled_rate_p95`), keeping at
least `place_field_min_bins` bins. We outline it on the unit's rate map alongside
its events on the trajectory.

In [ ]:
from camap.analysis.spatial_2d import compute_place_field_mask, _seed_rate_map
from matplotlib.collections import LineCollection

field = compute_place_field_mask(
    _seed_rate_map(res),
    threshold=ds.spatial.place_field_threshold,
    min_bins=ds.spatial.place_field_min_bins,
    shuffled_rate_p95=res.shuffled_rate_p95,
)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.6))

# Left: trajectory (faint) + events (alpha ~ amplitude/occupancy).
x = ds.trajectory_filtered["x"].to_numpy(); y = ds.trajectory_filtered["y"].to_numpy()
pts = np.column_stack([x, y]).reshape(-1, 1, 2)
segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
axL.add_collection(LineCollection(segs, colors=[(0, 0, 0, 0.1)] * len(segs), linewidths=0.3))
vis = res.vis_data_above
if vis is not None and len(vis):
    xb = np.clip(np.digitize(vis["x"], ds.x_edges) - 1, 0, len(ds.x_edges) - 2)
    yb = np.clip(np.digitize(vis["y"], ds.y_edges) - 1, 0, len(ds.y_edges) - 2)
    na = vis["s"].to_numpy() / np.maximum(ds.occupancy_time[xb, yb], 0.01)
    axL.scatter(vis["x"], vis["y"], c="red", s=80,
                alpha=np.clip(na / (na.max() or 1), 0, 1), edgecolors="none")
axL.set_xlim(ext[0], ext[1]); axL.set_ylim(ext[2], ext[3])
axL.set_aspect("equal"); axL.axis("off"); axL.set_title(f"Unit {UID}: trajectory + events")

# Right: rate map + place-field contour.
disp = res.rate_map_smoothed.copy(); disp[~ds.valid_mask] = np.nan
im = axR.imshow(disp.T, origin="lower", extent=ext, cmap="inferno", aspect="equal")
if np.any(field):
    axR.contour(field.T.astype(float), levels=[0.5], colors="white",
                linewidths=1.2, extent=ext, origin="lower")
axR.axis("off"); axR.set_title(f"rate map + place field  (SI={res.si:.2f}, p={res.p_val:.3f})")
fig.colorbar(im, ax=axR, fraction=0.046, label="rate (a.u./s)")
plt.tight_layout(); plt.show()

## 10 · Gallery — the place cells we found

Top row: each cell's events on the trajectory. Bottom row: its rate map with the
place-field contour. A shared color scale makes peak rates comparable across
cells.

In [ ]:
rng = np.random.default_rng(0)
pool_ids = list(place.keys()) if place else list(ds.unit_results.keys())
gallery = sorted(rng.choice(pool_ids, size=min(4, len(pool_ids)), replace=False))
print("gallery units:", gallery)

maps = []
for uid in gallery:
    m = ds.unit_results[uid].rate_map_smoothed.copy(); m[~ds.valid_mask] = np.nan
    maps.append(m)
vmax = max(np.nanmax(m) for m in maps) if maps else 1.0
n = len(gallery)
fig, axes = plt.subplots(2, n, figsize=(3.2 * n + 0.8, 6), squeeze=False)

for col, uid in enumerate(gallery):
    r = ds.unit_results[uid]
    # trajectory + events
    at = axes[0, col]
    at.add_collection(LineCollection(segs, colors=[(0, 0, 0, 0.1)] * len(segs), linewidths=0.3))
    v = r.vis_data_above
    if v is not None and len(v):
        xb = np.clip(np.digitize(v["x"], ds.x_edges) - 1, 0, len(ds.x_edges) - 2)
        yb = np.clip(np.digitize(v["y"], ds.y_edges) - 1, 0, len(ds.y_edges) - 2)
        na = v["s"].to_numpy() / np.maximum(ds.occupancy_time[xb, yb], 0.01)
        at.scatter(v["x"], v["y"], c="red", s=70,
                   alpha=np.clip(na / (na.max() or 1), 0, 1), edgecolors="none")
    at.set_xlim(ext[0], ext[1]); at.set_ylim(ext[2], ext[3])
    at.set_aspect("equal"); at.axis("off"); at.set_title(f"Cell {uid}", fontsize=9)
    # rate map + field
    ar = axes[1, col]
    im = ar.imshow(maps[col].T, origin="lower", extent=ext, cmap="inferno",
                   aspect="equal", vmin=0, vmax=vmax)
    fld = compute_place_field_mask(_seed_rate_map(r),
                                   threshold=ds.spatial.place_field_threshold,
                                   min_bins=ds.spatial.place_field_min_bins,
                                   shuffled_rate_p95=r.shuffled_rate_p95)
    if np.any(fld):
        ar.contour(fld.T.astype(float), levels=[0.5], colors="white",
                   linewidths=1, extent=ext, origin="lower")
    ar.axis("off"); ar.set_title(f"SI={r.si:.2f} p={r.p_val:.3f}", fontsize=8)
fig.tight_layout(rect=[0, 0, 0.92, 1])
cax = fig.add_axes([0.94, axes[1, 0].get_position().y0, 0.015,
                    axes[1, 0].get_position().height])
fig.colorbar(im, cax=cax, label="rate (a.u./s)")
plt.show()

## 11 · Explore — interactive unit browser

Every unit, with full detail panels (footprint, trajectory+events, the three
rate maps, SI histogram, neural trace). Use the toggle to switch between all
units and place cells only. **Needs Jupyter Lab** (`%matplotlib widget`).

In [ ]:
%matplotlib widget
from camap.notebook import browse_units
from IPython.display import display

fig, controls = browse_units(ds)
plt.show()
display(controls)

## 12 · `save_bundle()` — persist results

Writes a self-contained `.camap` directory (config, arrays, the canonical table,
per-unit results, and summary figures as PDFs). Reopen it later — without
re-running anything — in CaMAP's `notebook/view_results_arena.ipynb` viewer, or
with `BaseCaMAPDataset.load_bundle(path)`.

In [ ]:
bundle = ds.save_bundle("output/workshop_demo")
print("Saved", bundle)